# Inferencia Estadística TP4

**Eduardo Nicolas Sanchez Lopez**

Costo de oportunidad del inventario de bolsas de fibra de 600 gramos. Ejecutar las celdas de arriba hacia abajo. Los importes monetarios se presentan con dos decimales; los cálculos conservan su precisión original.

In [1]:
import sys
from pathlib import Path

PROJECT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src/analysis.py").is_file()
)
sys.path.insert(0, str(PROJECT / "src"))
print("Entorno local preparado.")

Entorno local preparado.


## Datos y preparación del cálculo

La planilla se verifica mediante SHA-256. Se usan datos históricos fechados, no cotizaciones actuales. Las explicaciones y tablas se generan desde la misma plantilla que el informe PDF.

In [2]:
import json
import math
import re
from IPython.display import HTML, display
from analysis import analyze, opportunity_cost, value_inventory
from build_report import build_report

result = analyze(PROJECT)
report_paths = build_report(PROJECT)
report_html = report_paths["html"].read_text()
sections = re.findall(r"(<h2>.*?)(?=<h2>|</body>)", report_html, flags=re.S)
assert len(sections) == 8
sources = json.loads((PROJECT / "data/reference/sources.json").read_text())["sources"]

notebook_style = """
<style>
.tp4-output {font-family:Arial,sans-serif; font-size:15px; line-height:1.5; max-width:980px; color:#17212b; background:white; padding:18px;}
.tp4-output h2,.tp4-output h3 {color:#174f7a;}
.tp4-output table {width:100%; border-collapse:collapse; margin:16px 0;}
.tp4-output th {background:#174f7a; color:white; text-align:left;}
.tp4-output td,.tp4-output th {border:1px solid #cbd5df; padding:8px;}
.tp4-output .num {text-align:right;}
.tp4-output .equation {text-align:center; padding:12px; line-height:1.8;}
.tp4-output .note,.tp4-output .result {background:#eaf3f9; border-left:3px solid #174f7a; padding:12px; margin:12px 0;}
.tp4-output .caption {font-size:13px; color:#536273;}
.tp4-output a {color:#174f7a;}
</style>
"""

def show_section(number):
    fragment = re.sub(r"</?section\b[^>]*>", "", sections[number - 1])
    for source in sources:
        fragment = fragment.replace(f'href="#source-{source["id"]}"', f'href="{source["url"]}"')
    display(HTML(notebook_style + '<div class="tp4-output">' + fragment + '</div>'))

print("Planilla verificada y datos históricos comparables. Cálculo sin redondeos intermedios.")

Planilla verificada y datos históricos comparables. Cálculo sin redondeos intermedios.


In [3]:
# 1. Costo de adquisición por bolsa, antes de ganancia e IVA de venta.
unit_cost_ars = result["pallet_cost_ars"] / result["pallet_bags"]
assert math.isclose(unit_cost_ars, result["unit_cost_ars"])
show_section(1)

Componente informado,Celda,Importe por palet (ARS)
Precio de fábrica (palet de 1.260 bolsas),E59,"5.490.190,00"
"Impuesto a las ganancias, según planilla",E60,"1.811.764,00"
"Impuesto a los ingresos brutos, según planilla",E61,"164.705,00"
Descarga,E62,"100.000,00"
Seguro,E63,"7.566,00"
Costo puesto en Mendoza,E65,"7.574.225,00"


In [4]:
# 2. Conversión con el MEP fechado.
unit_cost_usd = unit_cost_ars / result["exchange_rate"]["ars_per_usd"]
assert math.isclose(unit_cost_usd, result["unit_cost_usd"])
show_section(2)

In [5]:
# 3. Media de estados corregidos, no media ponderada por días.
corrected_balances = [row["corrected_balance"] for row in result["stock_audit"]]
mean_stock = sum(corrected_balances) / len(corrected_balances)
stock_value_usd = value_inventory(mean_stock, unit_cost_ars, result["exchange_rate"]["ars_per_usd"])
assert math.isclose(stock_value_usd, result["stock_value_usd"])
show_section(3)

Verificación de la reconstrucción,Bolsas
Stock inicial + compras − ventas,3.396 + 47.880 − 47.528
Stock final corregido,3.748
Promedio de los 37 estados de inventario,"4.112,51"


In [6]:
# 4. Misma moneda, período, fecha y base NAV para ambos fondos.
annual_returns = {fund["ticker"]: fund["annual_return"] for fund in result["funds"]}
show_section(4)

Fondo,Período y base comunes,Retorno anual
AOR,Un año al 31/05/2025 · NAV · USD,"9,901322 % [1]"
SGOV,Un año al 31/05/2025 · NAV · USD,"4,780024 % [2]"


In [7]:
# 5. Escenarios anuales excluyentes: no se suman.
annual_costs = {ticker: opportunity_cost(stock_value_usd, rate) for ticker, rate in annual_returns.items()}
for fund in result["funds"]:
    assert math.isclose(annual_costs[fund["ticker"]], fund["annual_cost_usd"])
show_section(5)

Alternativa,Capital (USD),Tasa anual,Costo (USD/año)
AOR,"20.883,53","9,901322 %","2.067,75"
SGOV,"20.883,53","4,780024 %","998,24"


In [8]:
show_section(6)

## Actividad extra

La regla de inventario se presenta simbólicamente. No se fijan una cobertura empresarial ni una cantidad numérica sin validar los meses, la demanda y el plazo de entrega.

In [9]:
show_section(7)

In [10]:
show_section(8)

## Archivos generados

El PDF, el HTML y los resultados JSON se encuentran en `reports/generated/`. En Colab pueden descargarse desde el panel **Archivos**. Los cambios hechos en Colab no se guardan automáticamente en GitHub; para conservarlos se utiliza **Archivo → Guardar una copia en GitHub** o **Guardar una copia en Drive**.

In [11]:
# En Colab se descarga el PDF; localmente se indican las rutas relativas.
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(report_paths["pdf"]))
else:
    for path in report_paths.values():
        print(f"Archivo generado: {path.relative_to(PROJECT)}")

Archivo generado: reports/generated/TP4_Inferencia_Estadistica.html
Archivo generado: reports/generated/TP4_Inferencia_Estadistica.pdf
